In [11]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

%cd /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/

/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis


In [12]:
VAL_RESULTS_PATH_RAW = 'thesis-figures/extended_cbm/results/1_extended_cbm_results.parquet'

In [13]:
def create_summary_table(df: pd.DataFrame, filter_tag:list[str], reference_measure:str):
    """Erstellt eine Tabelle mit den Bestwerten pro Run."""
    print("\nErstelle Summary Tabelle (Max-Werte)...")

    # filter df for each column where df[column] is true
    for tag in filter_tag:
        col = f"tag_{tag}"
        if col not in df.columns:
            raise ValueError(f"Tag-Spalte {col} existiert nicht im DataFrame.")
        df = df[df[col] == True]

    if df.empty:
        print("DataFrame nach Tag-Filter leer.")
        return df

    # Sicherstellen, dass innerhalb jedes Runs nach step sortiert ist
    df = df.sort_values(["run_id", "_step"]).copy()

    # Epoch als laufende Evaluation pro run_id (startet bei 0)
    df["epoch"] = df.groupby("run_id").cumcount()

    # Bestes F1-Concept pro Run auswählen
    best_per_run = (
        df.loc[df.groupby("run_id")[reference_measure].idxmax()]
        .reset_index(drop=True)
    )
    
    return best_per_run

In [14]:
# test_results = pd.read_parquet(TEST_RESULTS_PATH)
val_results_raw = pd.read_parquet(VAL_RESULTS_PATH_RAW)

val_results_joint = create_summary_table(val_results_raw, filter_tag=["joint"], reference_measure=r'$F_1$-Score Labels')


Erstelle Summary Tabelle (Max-Werte)...


In [15]:
val_results_joint.iloc[0]

_step                                                1798
Recall Concepts                                       0.0
Precision Concepts                                    0.0
Total Loss                                       1.672659
Mean IoU                                         0.552361
Foreground Dice                                   0.14603
$F_1$-Score Concepts                                  0.0
$F_1$-Score Labels                               0.848915
Mean Dice                                        0.562481
Concept Accuracy                                      0.0
Label Accuracy                                   0.848915
Runtime                                      11150.629892
normalized_epoch                                       24
epoch                                                  23
tag_Final                                            True
tag_aab-seg-benchmark                                None
run_id                                           2effny07
Concept Module

In [16]:
val_results_raw.columns

Index(['_step', 'Recall Concepts', 'Precision Concepts', 'Total Loss',
       'Mean IoU', 'Foreground Dice', '$F_1$-Score Concepts',
       '$F_1$-Score Labels', 'Mean Dice', 'Concept Accuracy', 'Label Accuracy',
       'Runtime', 'normalized_epoch', 'epoch', 'tag_Final',
       'tag_aab-seg-benchmark', 'run_id', 'Concept Module',
       'Segmentation Module', 'Concept Criterion', 'Use Soft Labels',
       'unified_model', 'Dataset', 'Concept Loss', 'tag_aab-con',
       'tag_aab-con-maskreg-intro', 'tag_aab-soft-bce', 'tag_aab-aff',
       'tag_joint', 'tag_con-implicit-es', 'tag_con-implicit-cs'],
      dtype='object')

In [17]:
from dataclasses import dataclass
from typing import Any

import torch


@dataclass
class EvalMetrics:
    # --- core eval metrics ---
    loss: float
    dice_mean: float
    iou_mean: float
    f1_concept_activations: float
    precision_concepts: float
    recall_concepts: float
    accuracy_concepts: float
    f1_labels: float
    accuracy_labels: float
    foreground_dice_scores: float

    # --- optional metadata (for dataframe concatenation later) ---
    concept_module: str | None = None
    segmentation_module: str | None = None
    dataset: str | None = None
    runtime: float | None = None

    # -------------------------
    # 2️⃣ Pretty renamed dict
    # -------------------------
    def to_pretty_dict(self) -> dict[str, Any]:
        return {
            "Total Loss": self.loss,
            "Mean Dice": self.dice_mean,
            "Mean IoU": self.iou_mean,
            r"Concept Activations $F_1$-Score": self.f1_concept_activations,
            "Precision Concepts": self.precision_concepts,
            "Recall Concepts": self.recall_concepts,
            "Concept Accuracy": self.accuracy_concepts,
            r"Label $F_1$-Score": self.f1_labels,
            "Label Accuracy": self.accuracy_labels,
            "Foreground Dice": self.foreground_dice_scores,
            "Concept Module": self.concept_module,
            "Segmentation Module": self.segmentation_module,
            "Dataset": self.dataset,
            "Runtime": self.runtime,
        }

    # -------------------------
    # 3️⃣ DataFrame row
    # -------------------------
    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([self.to_pretty_dict()])


In [18]:
import numpy as np
from tqdm import tqdm

from architecture.extended_cbm import CBMWrapper, ExtendedCBMOutput, init_from_checkpoint
from cbm_datasets import Batch, get_dataloader, get_datasets
from utils.others import calculate_metrics


@torch.no_grad()
def evaluate(
    model: CBMWrapper,
    val_loader: torch.utils.data.DataLoader,
    device: str,
    calculate_dice: bool,
    calculate_fg_dice: bool,
    calculate_iou: bool,
    concept_names: list[str],
    part_names: list[str],
    dice_thr: float = 0.5,
):
    model.eval()
    total_loss = 0.0
    total_samples = 0  # Trackt die tatsächliche Anzahl der Bilder

    dice_scores: list[float] = []
    iou_scores: list[float] = []
    foreground_dice_scores: list[float] = []
    f1_labels: list[float] = []

    all_concept_logits = []
    all_concept_targets = []

    all_label_preds = []
    all_label_targets = []


    correct = 0

    pbar = tqdm(val_loader, desc="[Eval]")
    for step, batch in enumerate(pbar):
        
        batch: Batch
        batch = batch.to(device)
        
        # Batch Größe ermitteln (wichtig für batch_size > 1)
        current_batch_size = batch.images.size(0)
        total_samples += current_batch_size

        predictions: ExtendedCBMOutput = model(batch.images)
        
        metrics = calculate_metrics(
            predictions,
            batch,
            mode="val",
            dice_thr=dice_thr,
            concept_names=concept_names,
            part_names=part_names,
            calculate_dice=calculate_dice,
            calculate_fg_dice=calculate_fg_dice,
            calculate_iou=calculate_iou,
            calculate_f1_concepts=False
        )

        # Accuracy Berechnung korrigiert für Batches
        pred_labels = predictions.classification_module.labels_logits.argmax(dim=1)
        gt_labels = batch.labels.argmax(dim=1)
        correct += (pred_labels == gt_labels).sum().item()

        # Metriken (calculate_metrics gibt meist schon Durchschnitte pro Batch zurück)
        if calculate_dice:
            dice_scores.append(metrics["val/dice_mean"])
        if calculate_fg_dice:
            foreground_dice_scores.append(metrics["val/foreground_dice_mean"])

        if calculate_iou:
            iou_scores.append(metrics["val/iou_mean"])
        
        all_concept_logits.append(
            predictions.concept_module.concept_logits.detach().cpu()
        )
        if batch.concepts is not None:
            all_concept_targets.append(
                batch.concepts.detach().cpu()
            )

        all_label_preds.append(pred_labels.cpu())
        all_label_targets.append(gt_labels.cpu())


    # Finale Berechnungen basierend auf total_samples statt len(val_loader)
    avg_loss = total_loss / total_samples

    avg_accuracy_labels = correct / total_samples

    # Da metrics meist Batch-Averages sind, ist np.mean über die Liste okay,
    # solange die Batches etwa gleich groß sind.
    avg_dice = float(np.mean(dice_scores)) if dice_scores else 0.0
    avg_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
    avg_foreground_dice = float(np.mean(foreground_dice_scores)) if foreground_dice_scores else 0.0

    if all_concept_logits:
        concept_logits = torch.cat(all_concept_logits, dim=0)
        concept_targets = torch.cat(all_concept_targets, dim=0)

        # Wahrscheinlichkeiten und Vorhersagen (Threshold 0.5)
        probs = torch.sigmoid(concept_logits)
        preds = (probs >= 0.5).int().cpu().numpy()
        targets = (concept_targets >= 0.5).int().cpu().numpy()

        # Wir berechnen 'macro', um jedes Konzept gleich zu gewichten
        prec_concepts = float(precision_score(targets, preds, average="macro", zero_division=0))
        rec_concepts = float(recall_score(targets, preds, average="macro", zero_division=0))
        f1_concepts = float(f1_score(targets, preds, average="macro", zero_division=0))

        acc_per_concept = (preds == targets).mean(axis=0)
        acc_concepts = float(acc_per_concept.mean())
    else:
        prec_concepts = rec_concepts = f1_concepts = acc_concepts = 0.0

    if all_label_preds:

        label_preds = torch.cat(all_label_preds, dim=0)  # [N,]
        label_targets = torch.cat(all_label_targets, dim=0)  # [N,]
        f1_labels = f1_score(label_targets, label_preds, average="micro") # type: ignore
    else:
        f1_labels = 0

    print(f"Correct: {correct} of {total_samples} (Acc: {avg_accuracy_labels:.4f})")

    
    return EvalMetrics(
        loss=avg_loss,
        dice_mean=avg_dice,
        iou_mean=avg_iou,
        f1_concept_activations=f1_concepts,
        precision_concepts=prec_concepts,
        recall_concepts=rec_concepts,
        accuracy_concepts=acc_concepts,
        f1_labels=f1_labels,
        accuracy_labels=avg_accuracy_labels,
        foreground_dice_scores=avg_foreground_dice,
    )
    
    

In [21]:
device = 'cuda'
run_id = '2effny07'
epoch = 23
dataset = 'CUB_112'

train_dataset, val_dataset, test_dataset, n_concepts, n_classes, _ = get_datasets(dataset_name=dataset, 
                                                            root_dir='/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/datasets', 
                                                            img_size=256, 
                                                            use_soft_labels=False, concept_masks_scale=None,
                                                            attr_level='image')

_, _, test_loader = get_dataloader(batch_size=4, num_workers=14, 
                                    train_dataset=train_dataset, val_dataset=val_dataset, 
                                    test_dataset=test_dataset)

model = init_from_checkpoint(run_id=run_id, epoch=epoch, dataset=dataset.lower(), device=device)
model = model.to(device)

results = evaluate(
    model=model,
    val_loader=test_loader,
    device=str(device),
    calculate_dice=False,
    calculate_fg_dice=False,
    calculate_iou=False,
    concept_names=[f"concept_{i}" for i in range(n_concepts)],
    part_names=[f"part_{i}" for i in range(n_classes)],
    dice_thr=0.5
)

Using cache found in /pfs/work9/workspace/scratch/ma_faroesch-master-thesis/.cache/hub/wimmerth_anyup_main
[Eval]: 100%|██████████| 1449/1449 [11:35<00:00,  2.08it/s]


Correct: 4973 of 5794 (Acc: 0.8583)
